In [ ]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D,Activation,LeakyReLU,BatchNormalization,MaxPooling2D,Flatten,Dense,Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator,img_to_array
import xgboost as xgb
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
import cv2
import os
from os import listdir
from pathlib import Path
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split

In [ ]:
# Set PLANT_DISEASE_DATA_DIR to the dataset root (the directory containing train/).
# The relative default keeps the notebook portable and does not change the working directory.
root_dir = Path(os.environ.get("PLANT_DISEASE_DATA_DIR", "data/plantvillage")).expanduser()
data_dir = root_dir / "train"
if not data_dir.is_dir():
    raise FileNotFoundError(
        f"Training directory not found: {data_dir}. Set PLANT_DISEASE_DATA_DIR to the dataset root."
    )


In [ ]:
# Dimension of resized image
DEFAULT_IMAGE_SIZE = (256, 256)
# Maximum number of images read from each class folder (existing notebook behavior).
N_IMAGES = 100

def convert_image_to_array(image_path):
    image = cv2.imread(os.fspath(image_path))
    if image is None:
        raise ValueError(f"Could not decode image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, DEFAULT_IMAGE_SIZE)
    return img_to_array(image)


In [ ]:
print("Load images from all classes ...")
plant_disease_folder_list = listdir(data_dir)
print(len(plant_disease_folder_list))

# Loading Training & Validation Data

In [ ]:
image_list, label_list = [], []
class_dirs = sorted(path for path in Path(data_dir).iterdir() if path.is_dir())
if not class_dirs:
    raise ValueError(f"No class subdirectories found in {data_dir}")

for class_dir in class_dirs:
    image_paths = sorted(
        path for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    if not image_paths:
        raise ValueError(f"No supported image files found in {class_dir}")
    for image_path in image_paths[:N_IMAGES]:
        image_list.append(convert_image_to_array(image_path))
        label_list.append(class_dir.name)

if not image_list:
    raise ValueError(f"No images found under {data_dir}")
print(f"Loaded {len(image_list)} images across {len(class_dirs)} classes.")


In [ ]:
print(len(image_list))

In [ ]:
# Transform the loaded training image data into numpy array
np_image_list = np.array(image_list, dtype=np.float16) / 255.0

# Check the number of images loaded for training
image_len = len(image_list)
print(f"Total number of images: {image_len}")

# One-Hot Encoding the labels

In [ ]:
"""Encode classes as one-hot vectors (including the two-class case)."""
label_binarizer = LabelBinarizer()
encoded_labels = label_binarizer.fit_transform(label_list)
n_classes = len(label_binarizer.classes_)
if n_classes < 2:
    raise ValueError("At least two class folders are required for classification.")
if n_classes == 2:
    image_labels = np.column_stack((1 - encoded_labels[:, 0], encoded_labels[:, 0]))
else:
    image_labels = encoded_labels
print(f"Loaded {n_classes} classes.")


In [ ]:
"""# Augment and Split Dataset
Using `ImageDataGenerator` to augment data by performing various operations on the training images.
"""

augment = ImageDataGenerator(rotation_range=25, width_shift_range=0.1,
                             height_shift_range=0.1, shear_range=0.2, 
                             zoom_range=0.2, horizontal_flip=True, 
                             fill_mode="nearest")

In [ ]:
"""Create deterministic, class-stratified image-level train/validation/test splits.

This is not group-aware: near-duplicate images or multiple photos of one plant may
still cross partitions when the dataset has no plant/session identifiers.
"""
class_ids = np.argmax(image_labels, axis=1)
class_counts = np.bincount(class_ids, minlength=n_classes)
if class_counts.min() < 6:
    raise ValueError(
        "At least 6 readable images per class are required for the stratified "
        "70/15/15 split; inspect the dataset or revise the split plan."
    )

x_train, x_holdout, y_train, y_holdout, holdout_classes = train_test_split(
    np_image_list,
    image_labels,
    class_ids,
    test_size=0.30,
    random_state=42,
    stratify=class_ids,
)
x_valid, x_test, y_valid, y_test = train_test_split(
    x_holdout,
    y_holdout,
    holdout_classes,
    test_size=0.5,
    random_state=42,
    stratify=holdout_classes,
)
print(f"Split sizes: train={len(x_train)}, validation={len(x_valid)}, test={len(x_test)}")


In [ ]:
"""# Build Model
Defining the hyperparameters of the plant disease classification model.
"""

EPOCHS = 10
STEPS = 100
LR = 1e-3
BATCH_SIZE = 32
WIDTH = 256
HEIGHT = 256
DEPTH = 3


In [ ]:
"""Creating a Sequential Model to build CNN for multi-class classification"""

model = Sequential()
inputShape = (HEIGHT, WIDTH, DEPTH)
chanDim = -1

if K.image_data_format() == "channels_first":
    inputShape = (DEPTH, HEIGHT, WIDTH)
    chanDim = 1

model.add(Conv2D(32, (3, 3), padding="same",input_shape=inputShape))
model.add(LeakyReLU(alpha=0.1))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(3, 3)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), padding="same"))
model.add(LeakyReLU(alpha=0.1))
model.add(BatchNormalization(axis=chanDim))

model.add(Conv2D(64, (3, 3), padding="same"))
model.add(LeakyReLU(alpha=0.1))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), padding="same"))
model.add(LeakyReLU(alpha=0.1))
model.add(BatchNormalization(axis=chanDim))

model.add(Conv2D(128, (3, 3), padding="same"))
model.add(LeakyReLU(alpha=0.1))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(1024, name = 'my_dense'))
model.add(LeakyReLU(alpha=0.1))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(n_classes))
model.add(Activation("softmax"))

model.summary()

## Feature Extraction from the Dense Layer

In [ ]:
layer_name = 'my_dense'
intermediate_layer_model = Model(inputs=model.input,outputs=model.get_layer(layer_name).output) 
intermediate_layer_model.summary()

In [ ]:
"""# Train Model
We initialize Adam optimizer with learning rate and decay parameters. 
Also, we choose the type of loss and metrics for the model and compile it for training.
"""

# steps_per_epoch=len(x_train) // BATCH_SIZE,

# Initialize optimizer
opt = Adam(learning_rate=LR)

# Compile model
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])

# Train model
print("Training CNN...")
history = model.fit(augment.flow(x_train, y_train, batch_size=BATCH_SIZE),
                              validation_data=(x_valid, y_valid),
                              epochs=EPOCHS, 
                              verbose=1)
        


In [ ]:
"""# Evaluate Model
Comparing the accuracy and loss by plotting the graph for training and validation.
"""

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

# Train and validation accuracy
plt.plot(epochs, acc, 'b', label='Training accurarcy')
plt.plot(epochs, val_acc, 'r', label='Validation accurarcy')
plt.title('Training and Validation accurarcy')
plt.legend()

plt.figure()

# Train and validation loss
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()
plt.show()




In [ ]:
# Extract features for model fitting and selection on the validation partition only.
x_train_predict = intermediate_layer_model.predict(x_train, verbose=0)
x_valid_predict = intermediate_layer_model.predict(x_valid, verbose=0)
print(x_valid_predict.shape)


# Support Vector Machine

Integration of CNN with Support Vector Machine.

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf')

svm.fit(x_train_predict,np.argmax(y_train,axis=1))

print('SVM Fit Complete')

In [ ]:
svm.score(x_train_predict,np.argmax(y_train,axis=1))

In [ ]:
svm.score(x_valid_predict, np.argmax(y_valid, axis=1))


In [ ]:
valid_predictions = svm.predict(x_valid_predict)
pd.DataFrame({
    "actual_class_id": np.argmax(y_valid, axis=1),
    "predicted_class_id": valid_predictions,
}).head()


# XGBoost Integration

Integrating the CNN model with Extreme Gradient Boost algorithm.

In [ ]:
import xgboost as xgb

xb = xgb.XGBClassifier(use_label_encoder=False)

xb.fit(x_train_predict,np.argmax(y_train,axis=1))

print('XGBoost Fit Complete')

#Save XGBoost Model
# pickle.dump(xb,open('xgb_model.pkl','wb'))

In [ ]:
xb.score(x_train_predict,np.argmax(y_train,axis=1))

In [ ]:
xb.score(x_valid_predict, np.argmax(y_valid, axis=1))


In [ ]:
# Choose this candidate based on validation results above, before running this cell.
# The test partition is held out for a single final assessment, not model selection.
selected_model = "cnn"  # one of: "cnn", "svm", "xgboost"
if selected_model == "cnn":
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"CNN held-out test loss: {test_loss:.4f}; accuracy: {test_accuracy:.4f}")
elif selected_model in {"svm", "xgboost"}:
    x_test_predict = intermediate_layer_model.predict(x_test, verbose=0)
    estimator = {"svm": svm, "xgboost": xb}[selected_model]
    test_accuracy = estimator.score(x_test_predict, np.argmax(y_test, axis=1))
    print(f"{selected_model} held-out test accuracy: {test_accuracy:.4f}")
else:
    raise ValueError("selected_model must be 'cnn', 'svm', or 'xgboost'")


# Further Scope of Improvement

- Image processing techniques can be implemented before approaching the CNN's input layer
- Developing a Real-Time PDD [Plant Disease Detection] system using Opencv
- Improving the Validation Loss 
- Wider scope of data augmentation & manipulation for better performance
- Integrating CNN with other architectures
- Hyperparameter tunning
- Additional Datasets
- Front End development [Django, Flask, HTML+CSS+JS, Kotlin, Java, Swift, Flutter]
- Cross Platform application development 
- Deploying the AI model onto Companion computers for real-time application 